# 03 · Vorverarbeitungs-Audit

Gespeicherte Artefakte für MedQA, MedMCQA und MMLU prüfen; kein Download oder Modell nötig.


In [1]:
# Projekt-Wurzel finden
import json
import pathlib
import sys
from collections import Counter

import pandas as pd
from IPython.display import display

PROJECT_ROOT = pathlib.Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PREPROCESSED_ROOT = PROJECT_ROOT / "outputs" / "preprocessed"
print("Artefakt-Wurzel:", PREPROCESSED_ROOT)

Artefakt-Wurzel: <Projektordner>/analysis/outputs/preprocessed


## Artefakte laden

Pro Datensatz wird das neueste Zeitstempel-Verzeichnis mit `manifest.json` gewählt.


In [2]:
def latest_artifact(alias):
    candidates = sorted(
        path.parent
        for path in (PREPROCESSED_ROOT / alias).glob("*/manifest.json")
    )
    if not candidates:
        raise FileNotFoundError(
            f"Kein Artefakt für {alias}. Zuerst scripts/preprocess_data.py ausführen."
        )
    return candidates[-1]

artifacts = {alias: latest_artifact(alias) for alias in ("medqa", "medmcqa", "mmlu")}
for alias, path in artifacts.items():
    print(f"{alias:8s} -> {path.name}")

medqa    -> 20260911T083840Z_test_mcq-preprocessing-v1
medmcqa  -> 20260911T083843Z_validation_mcq-preprocessing-v1
mmlu     -> 20260911T083847Z_test_mcq-preprocessing-v1


## Population

Ausschlüsse zählen eindeutige Zeilen; mehrere Gründe pro Zeile sind möglich.


In [3]:
manifests = {
    alias: json.loads((path / "manifest.json").read_text())
    for alias, path in artifacts.items()
}

overview_rows = []
for alias, manifest in manifests.items():
    overview_rows.append({
        "Datensatz": alias,
        "Split": manifest["dataset_split"],
        "Rohzeilen": manifest["raw_rows"],
        "Akzeptiert": manifest["selected_samples"],
        "Ausgeschlossen": manifest["excluded_rows"],
        "Akzeptiert (%)": round(100 * manifest["selected_samples"] / manifest["raw_rows"], 2),
        "HF-Revision": manifest.get("dataset_revision_resolved"),
        "Fingerprint": manifest.get("dataset_fingerprint"),
    })

overview = pd.DataFrame(overview_rows)
display(overview)

,Datensatz,Split,Rohzeilen,Akzeptiert,Ausgeschlossen,Akzeptiert (%),HF-Revision,Fingerprint
0,medqa,test,1273,1273,0,100.00,20a8f4d6b851f6391751f6e76c06bc3a26c83e0b,9e4e5c3c66e6bc35
1,medmcqa,validation,4183,3941,242,94.21,91c6572c454088bf71b679ad90aa8dffcd0d5868,0598613f1e3b097f
2,mmlu,test,14042,13581,461,96.72,c30699e8356da336a370243923dbaf21066bb9fe,436299c1c09696bb


## Integrität

SHA-256-Prüfsummen, Zeilenzahlen, eindeutige IDs und vier Permutationen pro Sample prüfen.


In [4]:
from src.data.preprocessing import load_preprocessed_artifact

frames = {}
integrity_rows = []
for alias, artifact in artifacts.items():
    samples, manifest = load_preprocessed_artifact(artifact, verify_hashes=True)
    sample_df = pd.read_parquet(artifact / "samples.parquet")
    prompt_df = pd.read_parquet(artifact / "cyclic_prompts.parquet")
    excluded_df = pd.read_parquet(artifact / "excluded_rows.parquet")
    frames[alias] = {"samples": sample_df, "prompts": prompt_df, "excluded": excluded_df}

    prompt_counts = prompt_df.groupby("sample_id").size()
    permutation_sets = prompt_df.groupby("sample_id")["permutation_id"].apply(
        lambda values: set(values.astype(int))
    )
    integrity_rows.append({
        "Datensatz": alias,
        "Checksums": "OK",
        "Sample-IDs eindeutig": sample_df["sample_id"].is_unique,
        "4 Prompts je Sample": prompt_counts.eq(4).all(),
        "Permutations-IDs 0–3": permutation_sets.map(lambda x: x == {0, 1, 2, 3}).all(),
        "Promptzeilen": len(prompt_df),
        "Erwartet": 4 * len(sample_df),
    })

display(pd.DataFrame(integrity_rows))

,Datensatz,Checksums,Sample-IDs eindeutig,4 Prompts je Sample,Permutations-IDs 0–3,Promptzeilen,Erwartet
0,medqa,OK,True,True,True,5092,5092
1,medmcqa,OK,True,True,True,15764,15764
2,mmlu,OK,True,True,True,54324,54324


## Ausschlussgründe

Positionsverweise, Meta-Antworten, doppelte Optionstexte, ungültige Rohzeilen und wiederholte Sample-IDs.


In [5]:
reason_rows = []
for alias, data in frames.items():
    counts = Counter(
        reason
        for reasons in data["excluded"].get("reason_codes", [])
        for reason in reasons
    )
    for reason, count in sorted(counts.items()):
        reason_rows.append({"Datensatz": alias, "Grund": reason, "Anzahl": count})

reason_table = pd.DataFrame(reason_rows)
if len(reason_table):
    display(reason_table.pivot(index="Grund", columns="Datensatz", values="Anzahl").fillna(0).astype(int))
else:
    print("Keine Ausschlüsse in den geladenen Artefakten.")

Datensatz,medmcqa,mmlu
Grund,,
duplicate_option_text,21,14
option_label_combination,2,64
relative_position_reference,219,391


## Ausgeschlossene Beispiele


In [6]:
for alias in ("medmcqa", "mmlu"):
    excluded = frames[alias]["excluded"]
    print(f"\n=== {alias.upper()} ===")
    all_reasons = sorted({reason for reasons in excluded["reason_codes"] for reason in reasons})
    for reason in all_reasons:
        subset = excluded[excluded["reason_codes"].map(lambda values: reason in values)].head(3)
        print(f"\n{reason} ({len(excluded[excluded['reason_codes'].map(lambda values: reason in values)])} Zeilen)")
        display(subset[["source_index", "question", "options"]])


=== MEDMCQA ===

duplicate_option_text (21 Zeilen)


,source_index,question,options
1,44,Multiple canals in mandibular premolars are se...,"[Africas, Caucians, Not Recalled, Not Recalled]"
6,161,Tonsils developed from:,"[Ventral part of 3rd pouch., Ventral part of 2..."
10,205,According to WHO classiﬁcation of periapical p...,"[4.6, 4.6, 4.61, 4.62]"



option_label_combination (2 Zeilen)


,source_index,question,options
98,1696,Cysts associated with vital teeth are:,"[Dentigerous cyst, Globulomaxillary cyst, late..."
231,3873,The largest commissure of the brain:,"[Commissure of Ganser, Commissure of Gudden, C..."



relative_position_reference (219 Zeilen)


...

,source_index,question,options
10,389,______ are the obligations of workers towards ...,"[Employee rights, Employee rights, Employer du..."
26,1941,The energy stored in the magnetic field in a s...,"[0.015 joule., 0.15 joule., 0.015 joule., 1.15..."
60,2178,"Subtract. 2,396 – 1,709","[687, 687, 1,493, 1,695]"



option_label_combination (64 Zeilen)


,source_index,question,options
1,239,Which of the following is/are true?,[Titan is the only outer solar system moon wit...
2,253,Which of the following is/are common feature(s...,"[ejecta, raised rims, central peaks, A and B o..."
3,267,Venus shows evidence of which of the following...,"[Impacts, Erosion, Volcanism, A B and C]"



relative_position_reference (391 Zeilen)


,source_index,question,options
0,150,Which of the following bony structures may be ...,"[The supraorbital foramen, The infraorbital fo..."
4,268,Planetary rings are,[known to exist for all of the jovian planets....
6,324,Moons cause/contribute to which of the following?,"[stability of particles within rings., gravita..."


## Inhaltliche Dubletten

Identische Fragen und Optionen mit verschiedenen IDs bleiben erhalten und werden im Manifest gezählt.


In [7]:
duplicate_audit = pd.DataFrame([
    {
        "Datensatz": alias,
        "Dubletten-Gruppen behalten": manifest["exact_duplicate_content_groups_retained"],
        "Zusätzliche Dublettenzeilen behalten": manifest["exact_duplicate_content_extra_rows_retained"],
    }
    for alias, manifest in manifests.items()
])
display(duplicate_audit)

,Datensatz,Dubletten-Gruppen behalten,Zusätzliche Dublettenzeilen behalten
0,medqa,0,0
1,medmcqa,0,0
2,mmlu,102,102


## Vier zyklische Varianten

`display_to_content` ordnet sichtbare Positionen den ursprünglichen Inhaltsindizes zu; daraus folgt das jeweilige Goldlabel.


In [8]:
alias = "medqa"
sample_row = frames[alias]["samples"].iloc[0]
sample_id = sample_row["sample_id"]
permutations = frames[alias]["prompts"].query("sample_id == @sample_id").sort_values("permutation_id")

print("Frage:", sample_row["question"])
print("Originaler Goldindex:", sample_row["gold_index"], "/", sample_row["gold_label"])
display(permutations[[
    "permutation_id",
    "display_to_content",
    "displayed_option_A",
    "displayed_option_B",
    "displayed_option_C",
    "displayed_option_D",
    "gold_displayed_label",
]])

Frage: A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?
Originaler Goldindex: 1 / B


,permutation_id,display_to_content,displayed_option_A,displayed_option_B,displayed_option_C,displayed_option_D,gold_displayed_label
0,0,"[0, 1, 2, 3]",Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,Refuse to dictate the operative report,B
1,1,"[1, 2, 3, 0]",Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,Refuse to dictate the operative report,Disclose the error to the patient and put it i...,A
2,2,"[2, 3, 0, 1]",Report the physician to the ethics committee,Refuse to dictate the operative report,Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,D
3,3,"[3, 0, 1, 2]",Refuse to dictate the operative report,Disclose the error to the patient and put it i...,Tell the attending that he cannot fail to disc...,Report the physician to the ethics committee,C


## Abschlussprüfung


In [9]:
for alias, data in frames.items():
    samples = data["samples"]
    prompts = data["prompts"]
    manifest = manifests[alias]

    assert len(samples) == manifest["selected_samples"]
    assert samples["sample_id"].is_unique
    assert samples["gold_index"].between(0, 3).all()
    assert samples[["option_A", "option_B", "option_C", "option_D"]].notna().all().all()
    assert len(prompts) == 4 * len(samples)
    assert prompts["prompt_template_version"].eq(manifest["prompt_template_version"]).all()

print("✓ Alle drei Artefakte erfüllen die Übergabebedingungen.")
print("✓ Nächster Schritt: Modellscoring — noch keine Bias-Metrik und kein PriDe in diesem Notebook.")

✓ Alle drei Artefakte erfüllen die Übergabebedingungen.
✓ Nächster Schritt: Modellscoring — noch keine Bias-Metrik und kein PriDe in diesem Notebook.


Artefakte neu erzeugen: [RUN_PREPROCESSING.md](../RUN_PREPROCESSING.md). Für spätere Läufe das gewünschte Verzeichnis über `--preprocessed` angeben; beim Transfer über `--source-preprocessed` und `--target-preprocessed`.
